In [ ]:
import pathlib
import polars as pl

data_dir = pathlib.Path() / "data" / "playground-series-s5e12"

train = pl.read_csv(data_dir / "train.csv")
label = "diagnosed_diabetes"
X_train = train.select(pl.exclude(label))
y_train = train.select(pl.col(label))
X_test = pl.read_csv(data_dir / "test.csv")

numerical_columns = [
    "age",
    "alcohol_consumption_per_week",
    "physical_activity_minutes_per_week",
    "diet_score",
    "sleep_hours_per_day",
    "screen_time_hours_per_day",
    "bmi",
    "waist_to_hip_ratio",
    "systolic_bp",
    "diastolic_bp",
    "heart_rate",
    "cholesterol_total",
    "hdl_cholesterol",
    "ldl_cholesterol",
    "triglycerides",
]
categorical_columns = [
    "gender",
    "ethnicity",
    "education_level",
    "income_level",
    "smoking_status",
    "employment_status",
    "family_history_diabetes",
    "hypertension_history",
    "cardiovascular_history"
]

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.model_selection import StratifiedKFold, cross_val_score
from xgboost import XGBClassifier
from joblib import parallel_backend

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_columns),
        ("num", "passthrough", numerical_columns),
    ]
)
xgb = XGBClassifier()
pipeline = Pipeline(
    steps=[
        ("preprocess", preprocessor),
        ("model", xgb),
    ]
)
skf = StratifiedKFold(n_splits=7, shuffle=True, random_state=42)

with parallel_backend("threading"):
    scores = cross_val_score(
        pipeline,
        X_train.to_pandas(),
        y_train.to_pandas().iloc[:, 0],
        cv=skf,
        scoring="roc_auc",
        n_jobs=-1,
    )

print("CV scores:", scores)
print("Mean:", scores.mean(), "+/-", scores.std())


CV scores: [0.72257293 0.72280326 0.72277318 0.72033829 0.72257658 0.72546986
 0.722855  ]
Mean: 0.7227698730887429 +/- 0.0013777327804604584


In [31]:
with parallel_backend("threading"):
    pipeline.fit(X_train.to_pandas(), y_train.to_pandas().iloc[:, 0])
    preds = pipeline.predict_proba(X_test.to_pandas())[:, 1]

In [39]:
submission = pl.from_numpy(preds, {"diagnosed_diabetes":pl.Float64})
submission = submission.with_columns(
    X_test.select(pl.col("id")).to_series()
)

In [48]:
submission = (
    X_test
    .select(pl.col("id"))
    .with_columns(
        pl.Series("diagnosed_diabetes", preds)
    )
)
submission.write_csv(data_dir / "submission1.csv")